In [ ]:
import pandas as pd
import time
import webbrowser
import os
import json
from tqdm import tqdm

In [ ]:
def encode_dt(dt):
    if isinstance(dt, str):
        dt = pd.to_datetime(dt)
    return dt.strftime("%Y-%m-%d %H:%M").replace(" ", "%20")

def make_url(shipid, st, en):
    base = "https://www.marinetraffic.com/map/gettrackjson"
    return f"{base}/shipid:{shipid}/stdate:{encode_dt(st)}/endate:{encode_dt(en)}/trackorigin:livetrack"

In [ ]:
journeys_df = pd.read_csv('start_end_journeys.csv')
journeys_df.head()

# Map mmsi to shipid
mmsi_map = pd.read_csv('mmsi_shipid_map.csv').set_index('mmsi')['shipid'].to_dict()
journeys_df['shipid'] = journeys_df['mmsi'].map(mmsi_map)

In [ ]:
# Fetch coordinates for ship, group only by journey_id
# Open each journey in a new browser tab with a delay
# Files are downloaded from the opened tabs and saved locally

for journey_id, group in journeys_df.groupby('journey_id'):
    shipid = group['shipid'].iloc[0]
    start_time = group['timestamp'].min()
    end_time = group['timestamp'].max()
    url = make_url(shipid, start_time, end_time)
    print(f"Preparing to open for shipid={shipid}, journey_id={journey_id}: {url}")
    try:
        webbrowser.get('chrome').open_new_tab(url)
    except Exception as e:
        print(f"Error opening tab for journey_id={journey_id}: {e}")
    time.sleep(10)

### Combine coordinate jsons and merge them into a single file

In [ ]:
# Directory containing all JSON files
json_dir = "../archive/marineTraffic/all_jsons"
rows = []

# List all JSON files
json_files = [f for f in os.listdir(json_dir) if f.endswith(".json")]

for journey_id, filename in enumerate(tqdm(json_files, desc="Files"), start=1):
    # Extract shipid from filename (assuming format: shipid...json)
    shipid = filename.split("_")[0]
    file_path = os.path.join(json_dir, filename)
    with open(file_path, "r") as f:
        data = json.load(f)
        for entry in tqdm(data, desc=f"Entries in {filename}", leave=False):
            # Skip if not enough fields
            if len(entry) < 23:
                continue
            lat = float(entry[1])
            lon = float(entry[0])
            speed_kn = float(entry[2]) / 10
            course = entry[3]
            date = entry[5]
            source = entry[7]
            nav_status = entry[-2]
            draught = entry[-1]
            rows.append({
                "journey_id": journey_id,
                "shipid": shipid,
                "lat": lat,
                "lon": lon,
                "speed_kn": speed_kn,
                "course": course,
                "date": date,
                "source": source,
                "nav_status": nav_status,
                "draught": draught
            })

df = pd.DataFrame(rows)
print(df.head())

In [ ]:
# Map shipid back to mmsi using mmsi_shipid_map.csv
df['shipid'] = df['shipid'].astype(str)
mmsi_map = pd.read_csv('mmsi_shipid_map.csv')
mmsi_map['shipid'] = mmsi_map['shipid'].astype(str)
mmsi_dict = mmsi_map.set_index('shipid')['mmsi'].to_dict()
df['mmsi'] = df['shipid'].map(mmsi_dict)
df.head()

In [ ]:
# Export as pkl and csv
df.to_pickle("all_journeys.pkl")
df.to_csv("all_journeys.csv", index=False)

In [ ]:
df.sample()